# 04 — Enable online materialization and run online inference

Adapted from the official [4.Enable-online-store-run-inference.ipynb](https://github.com/Azure/azureml-examples/blob/65eceac7ff5cd2417af5f24d38192ab8e445dd85/sdk/python/featurestore_sample/notebooks/sdk_only/4.Enable-online-store-run-inference.ipynb) notebook
(MIT). The official code is kept as-is except for the local-execution changes listed in the
[README](README.md#changes-from-the-official-notebooks). Talk track: Slides 12, 20, and 23 in [FEATURE_STORE_TALK_TRACK.md](FEATURE_STORE_TALK_TRACK.md).

Create **Azure Managed Redis**, attach it as the online store, **materialize online on serverless Spark**, look up features **locally**, and serve them from a managed online endpoint. Azure Cache for Redis, which the official notebook creates, can no longer be created in this subscription.

| Label | Where the work runs |
|---|---|
| **[Local]** | Your computer: the notebook kernel, local Spark, or local Python |
| **[Control plane]** | Azure ML or Azure Resource Manager API call made from your computer |
| **[Serverless Spark]** | Azure ML managed Spark job submitted from your computer |

So far you have learned how to develop features, materialize them to offline materialization store, perform training, and perform batch inference. In this tutorial you will learn how to use feature store for online/realtime inference use cases.

You will perform the following steps:

1. Setup Azure Cache for Redis.
1. Attach the cache to feature store as the online materialization store and grant necessary permissions.
1. Materialize feature sets to the online store.
1. Test online deployment with mock data.

## Prerequisites

- Run the notebooks in order from this folder with the repository `.venv` kernel. Run
  [00_prepare_sample_data.ipynb](00_prepare_sample_data.ipynb) first.
- Sign in with `az login`. The notebooks use `AzureCliCredential` instead of `AzureMLOnBehalfOfCredential`,
  which works only inside Azure ML compute.
- Configure the feature store section of `workshop/.env`; see [README](README.md#configuration).
- Your account needs `Owner`, or `Contributor` plus `User Access Administrator`, on the resource group.

#### **[Local]** Load the local configuration
The official notebook runs in Azure ML studio, which provides the `AZUREML_ARM_*` environment variables. This cell provides the same values locally from `workshop/.env`. This notebook doesn't need Spark.

In [ ]:
# Local equivalent of the Azure ML studio Serverless Spark notebook session.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd().parents[1] / ".env", override=True)
os.environ["AZUREML_ARM_SUBSCRIPTION"] = os.environ["AZURE_SUBSCRIPTION_ID"]
os.environ["AZUREML_ARM_RESOURCEGROUP"] = os.environ["AZURE_RESOURCE_GROUP"]
os.environ["AZUREML_ARM_WORKSPACE_NAME"] = os.environ["AZUREML_WORKSPACE_NAME"]

# Local copy for local Spark development, and the ADLS Gen2 copy used by registered feature sets.
local_data_root = (Path.cwd().parents[1] / "outputs" / "feature_store_sample_data" / "feature-store-prp").as_uri()
cloud_data_root = (
    f"abfss://{os.environ['FEATURE_STORE_SOURCE_FILESYSTEM']}"
    f"@{os.environ['FEATURE_STORE_SOURCE_STORAGE_ACCOUNT']}.dfs.core.windows.net/feature-store-prp"
)
print("start spark session" if "spark" in globals() else "local configuration loaded")

### Setup root directory for the samples

In [ ]:
import os

# Local folder with the official featurestore_sample assets (featurestore/ and project/).
root_dir = os.path.join(os.getcwd(), "featurestore_sample")

if os.path.isdir(root_dir):
    print("The folder exists.")
else:
    print("The folder does not exist. Please create or fix the path")

### Initialize the project workspace CRUD client
The `MLClient` for the current workspace, where you are running this tutorial notebook, will be used for create, read, update, and delete (CRUD) operations.

In [ ]:
import os
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential

project_ws_sub_id = os.environ["AZUREML_ARM_SUBSCRIPTION"]
project_ws_rg = os.environ["AZUREML_ARM_RESOURCEGROUP"]
project_ws_name = os.environ["AZUREML_ARM_WORKSPACE_NAME"]

# Connect to the project workspace
ws_client = MLClient(
    AzureCliCredential(process_timeout=60), project_ws_sub_id, project_ws_rg, project_ws_name
)

### Initialize the CRUD client of the feature store workspace
The `MLClient` for the feature store workspace for create, read, update, and delete (CRUD) operations on feature store workspace.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential

# Feature store
featurestore_name = (
    os.environ["AZUREML_FEATURE_STORE_NAME"]  # use the same name from part #1 of the tutorial
)
featurestore_subscription_id = os.environ["AZUREML_ARM_SUBSCRIPTION"]
featurestore_resource_group_name = os.environ["AZUREML_ARM_RESOURCEGROUP"]

# Feature store MLClient
fs_client = MLClient(
    AzureCliCredential(process_timeout=60),
    featurestore_subscription_id,
    featurestore_resource_group_name,
    featurestore_name,
)

### Initialize the feature store core SDK client
This tutorial uses the Python feature store core SDK (`azureml-featurestore`). The SDK client initialized here is used for create, read, update, and delete (CRUD) operations, on feature stores, feature sets, and feature store entities.

In [ ]:
from azureml.featurestore import FeatureStoreClient
from azure.identity import AzureCliCredential

featurestore = FeatureStoreClient(
    credential=AzureCliCredential(process_timeout=60),
    subscription_id=featurestore_subscription_id,
    resource_group_name=featurestore_resource_group_name,
    name=featurestore_name,
)

## **[Control plane]** Set up the online store: Azure Managed Redis

The official notebook creates Azure Cache for Redis (Premium P2). Azure Cache for Redis is retiring, and creation is
rejected in this subscription:

> `BadRequest: Azure Cache for Redis is retiring, create Azure Managed Redis instance instead.`

Since April 1, 2026, Microsoft blocks new Basic, Standard, and Premium caches for tenants that had no Azure Cache for
Redis instance before that date. Tenants that had one can still create them until retirement on September 30, 2028
([What's new](https://learn.microsoft.com/azure/azure-cache-for-redis/cache-whats-new#july-2026)).

This notebook uses **Azure Managed Redis** (`Microsoft.Cache/redisEnterprise`), which Microsoft recommends. The
installed `azureml-featurestore` 1.2.2 online lookup client accepts Azure Managed Redis database IDs. The rest of
the official notebook is unchanged.

### Set values for the Azure Cache for Redis that will be used as online materialization store
In the following code cell, define the name of the Azure Cache for Redis that you want to create or reuse. Optionally, you can override other default settings.

In [ ]:
ws_location = ws_client.workspaces.get(ws_client.workspace_name).location

redis_subscription_id = os.environ["AZUREML_ARM_SUBSCRIPTION"]
redis_resource_group_name = os.environ["AZUREML_ARM_RESOURCEGROUP"]
redis_name = os.environ["FEATURE_STORE_REDIS_NAME"]
redis_location = ws_location

### Official option 1 (reference only): create Azure Cache for Redis
This is the official cell. It isn't run because Azure Cache for Redis creation is blocked for this tenant.

```python
from azure.mgmt.redis import RedisManagementClient
from azure.mgmt.redis.models import RedisCreateParameters, Sku, SkuFamily, SkuName

management_client = RedisManagementClient(
    AzureCliCredential(process_timeout=60), redis_subscription_id
)

# It usually takes about 5 - 10 min to finish the provision of the Redis instance.
# If the following begin_create() call still hangs for longer than that,
# please check the status of the Redis instance on the Azure portal and cancel the cell if the provision has completed.
# This sample uses a PREMIUM tier Redis SKU from family P, which may cost more than a STANDARD tier SKU from family C.
# Please choose the SKU tier and family according to your performance and pricing requirements.

redis_arm_id = (
    management_client.redis.begin_create(
        resource_group_name=redis_resource_group_name,
        name=redis_name,
        parameters=RedisCreateParameters(
            location=redis_location,
            sku=Sku(name=SkuName.PREMIUM, family=SkuFamily.P, capacity=2),
        ),
    )
    .result()
    .id
)

print(redis_arm_id)
```

### Create Azure Managed Redis
This cell creates the smallest Balanced tier (B0) with an OSS cluster database and sets `redis_arm_id`, the variable
the official cells use. Access keys stay enabled because the feature store's online lookup reads the database keys.

In [ ]:
import time

import requests
from azure.identity import AzureCliCredential

redis_subscription_id = os.environ["AZUREML_ARM_SUBSCRIPTION"]
redis_resource_group_name = os.environ["AZUREML_ARM_RESOURCEGROUP"]
managed_redis_name = redis_name
redis_location = os.environ["AZUREML_FEATURE_STORE_LOCATION"]
api_version = "2026-09-01"
cluster_id = (
    f"/subscriptions/{redis_subscription_id}/resourceGroups/{redis_resource_group_name}"
    f"/providers/Microsoft.Cache/redisEnterprise/{managed_redis_name}"
)


def arm_put_and_wait(resource_id: str, body: dict) -> dict:
    url = f"https://management.azure.com{resource_id}?api-version={api_version}"
    token = AzureCliCredential(process_timeout=60).get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {token}"}
    response = requests.put(url, json=body, headers=headers, timeout=60)
    if not response.ok:
        raise RuntimeError(f"{response.status_code}: {response.text}")
    while True:
        resource = requests.get(url, headers=headers, timeout=60).json()
        state = resource["properties"]["provisioningState"]
        if state in {"Succeeded", "Failed", "Canceled"}:
            print(f"{resource['name']}: {state}")
            return resource
        time.sleep(30)


# Smallest Balanced tier. Access keys stay enabled because the feature store's online lookup reads the keys.
cluster = arm_put_and_wait(cluster_id, {
    "location": redis_location,
    "sku": {"name": "Balanced_B0"},
    "properties": {"minimumTlsVersion": "1.2", "publicNetworkAccess": "Enabled"},
    "tags": {"SecurityControl": "Ignore"},
})
database = arm_put_and_wait(f"{cluster_id}/databases/default", {
    "properties": {
        "clientProtocol": "Encrypted",
        "port": 10000,
        "clusteringPolicy": "OSSCluster",
        "evictionPolicy": "NoEviction",
        "accessKeysAuthentication": "Enabled",
    }
})
redis_arm_id = database["id"]
print(redis_arm_id)

### Option 2: use an existing Azure Managed Redis instance
Optionally, reuse an existing Azure Managed Redis database with the previously defined name.

In [ ]:
redis_arm_id = "/subscriptions/{sub_id}/resourceGroups/{rg}/providers/Microsoft.Cache/redisEnterprise/{name}/databases/default".format(
    sub_id=redis_subscription_id,
    rg=redis_resource_group_name,
    name=redis_name,
)

# Step 1: **[Control plane]** Attach online materialization store to the feature store
Attach Azure Managed Redis to the feature store to be used as the online materialization store.

In [ ]:
from azure.ai.ml.entities import (
    ComputeRuntime,
    ManagedIdentityConfiguration,
    FeatureStore,
    MaterializationStore,
)

online_store = MaterializationStore(type="redis", target=redis_arm_id)

ml_client = MLClient(
    AzureCliCredential(process_timeout=60),
    subscription_id=featurestore_subscription_id,
    resource_group_name=featurestore_resource_group_name,
)

fs = FeatureStore(
    name=featurestore_name,
    online_store=online_store,
    compute_runtime=ComputeRuntime(spark_runtime_version="3.5"),
    tags={"SecurityControl": "Ignore"},
)

fs_poller = ml_client.feature_stores.begin_create(fs)
print(fs_poller.result())

#### Grant "Reader" role on the Azure Managed Redis cluster to the materialization identity
Attaching the online store grants the materialization identity **Contributor** on the online store target, which
for Azure Managed Redis is the **database**. The materialization and lookup client also reads the parent
**cluster** to find its host name, so without this role the backfill fails with
`AuthorizationFailed ... Microsoft.Cache/redisEnterprise/read`. This cell follows the role assignment pattern of
the official endpoint RBAC cells.

In [ ]:
from uuid import uuid4

from azure.core.exceptions import ResourceExistsError
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters

materialization_identity = ml_client.feature_stores.get(featurestore_name).materialization_identity
auth_client = AuthorizationManagementClient(AzureCliCredential(process_timeout=60), redis_subscription_id)
scope = redis_arm_id.split("/databases/")[0]

# The role definition ID for the "Reader" role.
role_definition_id = f"/subscriptions/{redis_subscription_id}/providers/Microsoft.Authorization/roleDefinitions/acdd72a7-3385-48ef-bd42-f606fba81ae7"
role_assignment_params = RoleAssignmentCreateParameters(
    principal_id=materialization_identity.principal_id,
    role_definition_id=role_definition_id,
    principal_type="ServicePrincipal",
)
try:
    auth_client.role_assignments.create(scope, str(uuid4()), role_assignment_params)
    print("Reader granted on the Azure Managed Redis cluster. Waiting two minutes for the role to propagate.")
    time.sleep(120)
except ResourceExistsError:
    print("Reader is already assigned on the Azure Managed Redis cluster.")

# Step 2: **[Serverless Spark]** Materialize `accounts` feature set data to online store

### Enable materialization on the `accounts` feature set
In the previous parts of the tutorial series, we did **not** materialize the accounts feature set since it had precomputed features and was used only for batch inference scenarios. In this step we will enable online materialization so that the features are available in the online store with low latency access. We will also enable offline materialization for consistency. Enabling offline materialization is optional.

In [ ]:
from azure.ai.ml.entities import (
    MaterializationSettings,
    MaterializationComputeResource,
)

# Turn on both offline and online materialization on the "accounts" featureset.

accounts_fset_config = fs_client._featuresets.get(name="accounts", version="1")

accounts_fset_config.materialization_settings = MaterializationSettings(
    offline_enabled=True,
    online_enabled=True,
    resource=MaterializationComputeResource(instance_type="standard_e8s_v3"),
    spark_configuration={
        "spark.driver.cores": 4,
        "spark.driver.memory": "36g",
        "spark.executor.cores": 4,
        "spark.executor.memory": "36g",
        "spark.executor.instances": 2,
    },
    schedule=None,
)

fs_poller = fs_client.feature_sets.begin_create_or_update(accounts_fset_config)
print(fs_poller.result())

### Backfill the `account` feature set
`backfill` command backfills data to all the materialization stores that are enabled for this feature set. In this case both offline and online materialization is enabled. Therefore `backfill` will be performed on both offline and online materialization stores.

In [ ]:
from datetime import datetime, timedelta, timezone

# Trigger backfill on the "accounts" feature set.
# Backfill from 01/01/2020 to all the way to 3 hours ago.

st = datetime(2020, 1, 1, 0, 0, 0, 0)
# Use UTC: a naive local time is treated as UTC by the SDK.
et = datetime.now(timezone.utc) - timedelta(hours=3)

poller = fs_client.feature_sets.begin_backfill(
    name="accounts",
    version="1",
    feature_window_start_time=st,
    feature_window_end_time=et,
    data_status=["None"],
)
print(poller.result().job_ids)

The following code cell tracks completion of the backfill job. Using the premium tier Azure Cache for Redis provisioned earlier, this step may take approximately 10 minutes to complete.

In [ ]:
# Get the job URL, and stream the job logs.
# With PREMIUM Redis SKU, SKU family "P", and cache capacity 2,
# it takes approximately 10 minutes to complete.
fs_client.jobs.stream(poller.result().job_ids[0])

# Step 3: **[Serverless Spark]** Materialize `transactions` feature set data to the online store

In the previous tutorials, we materialized data of the `transactions` feature set to the offline materialization store. In this step we will:

1. Enable online materilization for the `transactions` feature set.

In [ ]:
# Enable materialization to online store for the "transactions" feature set.

transactions_fset_config = fs_client._featuresets.get(name="transactions", version="1")
transactions_fset_config.materialization_settings.online_enabled = True
# The service returns the notebook 03 schedule start time as "04/15/2023 00:04:10 +00:00",
# then rejects that value on update. Set it back to the ISO 8601 string.
if transactions_fset_config.materialization_settings.schedule:
    transactions_fset_config.materialization_settings.schedule.start_time = "2023-04-15T00:04:10"

fs_poller = fs_client.feature_sets.begin_create_or_update(transactions_fset_config)
# print(fs_poller.result()) can't render the returned schedule start time as YAML.
materialization_settings = fs_poller.result().materialization_settings
print(f"offline: {materialization_settings.offline_enabled}, online: {materialization_settings.online_enabled}")

2. Backfill the data to both the online and offline materialization store to ensure that both have the latest data. Note that recurrent materialization job, which was setup earlier in the tutorial 3 of this series, will now materialize data to both online and offline materialization stores.

In [ ]:
# Trigger backfill on the "transactions" feature set to fill in the online/offline store.
# Backfill from 01/01/2020 to all the way to 4 hours ago.

from datetime import datetime, timedelta, timezone
from azure.ai.ml.entities import DataAvailabilityStatus

st = datetime(2020, 1, 1, 0, 0, 0, 0)
# Use UTC: a naive local time is treated as UTC by the SDK.
# With the notebook 03 schedule, the window must end before now - 3 hours (interval) - 20 minutes (source delay).
et = datetime.now(timezone.utc) - timedelta(hours=4)


poller = fs_client.feature_sets.begin_backfill(
    name="transactions",
    version="1",
    feature_window_start_time=st,
    feature_window_end_time=et,
    data_status=[DataAvailabilityStatus.NONE],
)
print(poller.result().job_ids)

The following code cell tracks completion of the backfill job. Using the premium tier Azure Cache for Redis provisioned earlier, this step may take approximately 5 minutes to complete.

In [ ]:
# Get the job URL, and stream the job logs.
# With PREMIUM Redis SKU, SKU family "P", and cache capacity 2,
# it takes approximately 5 minutes to complete.
fs_client.jobs.stream(poller.result().job_ids[0])

# Step 4: **[Local]** Test locally
In this step we will use our development environment (i.e. this notebook) to lookup features from online materialization store. 

First, we will parse the list of features from the existing feature retrieval specification:

In [ ]:
# Parse the list of features from the existing feature retrieval specification.
feature_retrieval_spec_folder = root_dir + "/project/fraud_model/feature_retrieval_spec"

features = featurestore.resolve_feature_retrieval_spec(feature_retrieval_spec_folder)

features

Next, we will get feature values from the online materialization store:

In [ ]:
from azureml.featurestore import init_online_lookup
import time

# Initialize the online store client.
init_online_lookup(features, AzureCliCredential(process_timeout=60))

Now, we will prepare some observation data for testing and use it to lookup features from the online materialization store. During online lookup, it may happen that the keys (`accountID`) defined in the observation sample data do not exist in the Redis (due to `TTL`). If this happens:
1. Open Azure portal.
2. Navigate to the Redis instance. 
3. Open console for the Redis instance and check for existing keys using command `KEYS *`.
4. Replace `accountID` values in the sample observation data with the existing keys.

In [ ]:
import pyarrow
from azureml.featurestore import get_online_features

# Prepare test observation data
obs = pyarrow.Table.from_pydict(
    {"accountID": ["A985156952816816", "A1055521248929430", "A914800935560176"]}
)

# Online lookup:
# It may happen that the keys defined in the observation sample data above does not exist in the Redis (due to TTL).
# If this happens, go to Azure portal and navigate to the Redis instance, open its console and check for existing keys using command "KEYS *"
# and replace the sample observation data with the existing keys.
df = get_online_features(features, obs)
df

Now that we have successfully looked up features from the online store, we will test online features using Azure Machine Learning managed online endpoint.

# Step 5: **[Control plane]** Test online features from Azure Machine Learning managed online endpoint
Managed online endpoint provide capability for deploying and scoring models for online/realtime inference. Optionally, you can use any inference technology of your choice (like kubernetes).

As a part of this step, we will perform the following actions:

1. Create an Azure Machine Learning managed online endpoint.
1. Grant required role-based access control (RBAC) permissions.
1. Deploy the model that we trained in the tutorial 3 of this tutorial series. The scoring script used in this step will have the code to lookup online features.
2. Perform scoring of the model with sample data. You will see that the online features are looked up and model scoring is completed successfully.

## Create Azure Machine Learning managed online endpoint
You can learn more about managed online endpoints [here](https://learn.microsoft.com/azure/machine-learning/how-to-deploy-online-endpoints?view=azureml-api-2&tabs=azure-cli). Note that using managed feature store API, you can also lookup online features from other inference platforms based on your need.

Following code defines a managed online endpoint with name `fraud-model`.

In [ ]:
from azure.ai.ml.entities import (
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
    Model,
    CodeConfiguration,
    Environment,
)


endpoint_name = os.environ["FEATURE_STORE_ONLINE_ENDPOINT_NAME"]

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name, auth_mode="key", tags={"SecurityControl": "Ignore"}
)

Excute the following code cell to create the managed online endpoint defined in the previous code cell.

In [ ]:
ws_client.online_endpoints.begin_create_or_update(endpoint).result()

## Grant required RBAC permissions
In this step, we will grant required RBAC permissions to the managed online endpoint on the Redis instance and feature store. The scoring code in the model deployment will need these RBAC permissions to successfully lookup features from the online store using the managed feature store API.

### Get managed identity of the managed online endpoint

In [ ]:
# Get managed identity of the managed online endpoint.
endpoint = ws_client.online_endpoints.get(endpoint_name)

model_endpoint_msi_principal_id = endpoint.identity.principal_id
model_endpoint_msi_principal_id

### Grant `Contributor` role to the online endpoint managed identity on the Azure Managed Redis 
We will grant `Contributor` role to the online endpoint managed identity on the Redis instance. This RBAC permission is needed to materialize data into the Redis online store.

In [ ]:
from azure.core.exceptions import ResourceExistsError
from azure.mgmt.msi import ManagedServiceIdentityClient
from azure.mgmt.msi.models import Identity
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters
from uuid import uuid4

auth_client = AuthorizationManagementClient(
    AzureCliCredential(process_timeout=60), redis_subscription_id
)

scope = f"/subscriptions/{redis_subscription_id}/resourceGroups/{redis_resource_group_name}/providers/Microsoft.Cache/redisEnterprise/{redis_name}"


# The role definition ID for the "contributor" role on the redis cache
# You can find other built-in role definition IDs in the Azure documentation
role_definition_id = f"/subscriptions/{redis_subscription_id}/providers/Microsoft.Authorization/roleDefinitions/b24988ac-6180-42a0-ab88-20f7382dd24c"

# Generate a random UUID for the role assignment name
role_assignment_name = str(uuid4())

# Set up the role assignment creation parameters
role_assignment_params = RoleAssignmentCreateParameters(
    principal_id=model_endpoint_msi_principal_id,
    role_definition_id=role_definition_id,
    principal_type="ServicePrincipal",
)

# Create the role assignment
try:
    # Create the role assignment
    result = auth_client.role_assignments.create(
        scope, role_assignment_name, role_assignment_params
    )
    print(
        f"Redis RBAC granted to managed identity '{model_endpoint_msi_principal_id}'."
    )
except ResourceExistsError:
    print(
        f"Redis RBAC already exists for managed identity '{model_endpoint_msi_principal_id}'."
    )

### Grant `AzureML Data Scientist` role to the online endpoint managed identity on the feature store
We will grant `AzureML Data Scientist` role to the online endpoint managed identity on the feature store. This RBAC permission is required for successful deployment of the model to the online endpoint.

In [ ]:
auth_client = AuthorizationManagementClient(
    AzureCliCredential(process_timeout=60), featurestore_subscription_id
)

scope = f"/subscriptions/{featurestore_subscription_id}/resourceGroups/{featurestore_resource_group_name}/providers/Microsoft.MachineLearningServices/workspaces/{featurestore_name}"

# The role definition ID for the "AzureML Data Scientist" role.
# You can find other built-in role definition IDs in the Azure documentation.
role_definition_id = f"/subscriptions/{featurestore_subscription_id}/providers/Microsoft.Authorization/roleDefinitions/f6c7c914-8db3-469d-8ca1-694a8f32e121"

# Generate a random UUID for the role assignment name.
role_assignment_name = str(uuid4())

# Set up the role assignment creation parameters.
role_assignment_params = RoleAssignmentCreateParameters(
    principal_id=model_endpoint_msi_principal_id,
    role_definition_id=role_definition_id,
    principal_type="ServicePrincipal",
)

# Create the role assignment
try:
    # Create the role assignment
    result = auth_client.role_assignments.create(
        scope, role_assignment_name, role_assignment_params
    )
    print(
        f"Feature store RBAC granted to managed identity '{model_endpoint_msi_principal_id}'."
    )
except ResourceExistsError:
    print(
        f"Feature store RBAC already exists for managed identity '{model_endpoint_msi_principal_id}'."
    )

### Grant `Storage Blob Data Reader` role to the online endpoint managed identity on the feature store default storage
The scoring script resolves the feature retrieval specification, which downloads each feature set specification
from the feature store's default storage. New feature stores use identity-based access for that storage
(`systemDatastoresAuthMode: identity`), so the endpoint identity needs **Storage Blob Data Reader** on it.
Without this role, the deployment fails in `init()` with `AuthorizationPermissionMismatch`. This cell follows
the pattern of the official RBAC cells above.

In [ ]:
import time

default_storage_id = ml_client.feature_stores.get(featurestore_name).storage_account

# The role definition ID for the "Storage Blob Data Reader" role.
role_definition_id = f"/subscriptions/{featurestore_subscription_id}/providers/Microsoft.Authorization/roleDefinitions/2a2b9908-6ea1-4ae2-8e65-a410df84e7d1"
role_assignment_params = RoleAssignmentCreateParameters(
    principal_id=model_endpoint_msi_principal_id,
    role_definition_id=role_definition_id,
    principal_type="ServicePrincipal",
)
try:
    auth_client.role_assignments.create(default_storage_id, str(uuid4()), role_assignment_params)
    print("Storage Blob Data Reader granted. Waiting two minutes for the role to propagate.")
    time.sleep(120)
except ResourceExistsError:
    print("Storage Blob Data Reader is already assigned.")

### Deploy the model to the online endpoint
First, inspect the scoring script `project/fraud_model/online_inference/src/scoring.py`. The scoring script performs the following tasks:

1. Load the feature metadata from the feature retrieval specification that was packaged along with the model during model training (tutorial 3 of this tutorial series). This specification has features from both `transactions` and `accounts` feature sets.
2. When an input inference request is received, the scoring code looks up the online features using the index keys from the request. In this case for both feature sets, the index column is the `accountID`.
3. Passes the features to the model to perform inference and returs the response, a boolean value representing the variable `is_fraud`.

First, create managed online deployment definition for model deployment by executing the following code cell:

In [ ]:
deployment = ManagedOnlineDeployment(
    name="green",
    endpoint_name=endpoint_name,
    model="azureml:fraud_model:1",
    code_configuration=CodeConfiguration(
        code=root_dir + "/project/fraud_model/online_inference/src/",
        scoring_script="scoring.py",
    ),
    environment=Environment(
        conda_file=root_dir + "/project/fraud_model/online_inference/conda.yml",
        # Ubuntu 20.04 (glibc 2.31) can't load the azureml-dataprep-rslex build that azureml-featurestore imports.
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04",
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)

Next, deploy the model to online enpoint by executing the following code cell. Note that it may take approximately 4-5 minutes to deploy the model.

In [ ]:
# Model deployment to online enpoint may take 4-5 minutes.
ws_client.online_deployments.begin_create_or_update(deployment).result()

### Test online deployment with mock data
Finally, execute the following code to test the online deployment using the mock data. You should see `0` or `1` as the output of this cell.

In [ ]:
# Test the online deployment using the mock data.
sample_data = root_dir + "/project/fraud_model/online_inference/test.json"
ws_client.online_endpoints.invoke(
    endpoint_name=endpoint_name, request_file=sample_data, deployment_name="green"
)

## Cleanup

Azure Managed Redis and the online endpoint incur cost until deleted. See [README](README.md#cleanup).